### Analyze $c_p$-distributions and $p^\prime_\mathrm{RMS}$

In [ ]:
import matplotlib.pyplot as plt
import torch as pt
from pandas import read_csv

from os import makedirs
from os.path import join, exists

from utils import compute_camber_line
from flowtorch.data import CSVDataloader

In [ ]:
# flow quantities
# u_inf = 238.845

# validation @ Ma = 0.73
u_inf = 242.16629

# angle(s) of attack
aoa = [3.5]

# chord length
chord = 1

# start for avg. cp-distributions
t_start = 0.6       # t = 0.6 -> tStart = 145 CTU

# use latex fonts
plt.rcParams.update({"text.usetex": True, "figure.dpi": 360, "text.latex.preamble": r"\usepackage{xcolor}"})

# use default color cycle
color = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f"]

# use these line styles
ls = ["-", "--", "-.", ":"]

In [ ]:
# define load and save paths
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation")
save_dir = join("..", "run", "plots", "DDES_URANS_validation_comparison_alpha_3.5deg")
cases = ["URANS_2D_Ma0.73_Re3e6/URANS_SA_alpha3.5deg_blockMesh", "URANS_2D_Ma0.73_Re3e6/URANS_SALSA_alpha3.5deg_blockMesh_useRmod_useSmod",
         "DDES_3D_Ma0.73_Re3e6/DDES_SA_Re3e6_Ma0.73_alpha3.5deg_y65_ymax0.25"]
legend = [r"$\mathrm{URANS}~(\mathrm{SA})$, $N_{\mathrm{cells}, y} = 1$", r"$\mathrm{URANS}~(\mathrm{SALSA})$, $N_{\mathrm{cells}, y} = 1$",
          r"$\mathrm{DDES}~(\mathrm{SA})$, $N_{\mathrm{cells}, y} = 65$"]

In [ ]:
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation", "URANS_2D_Ma0.73_Re3e6")
save_dir = join("..", "run", "plots", "URANS_validation", "URANS_blockMesh", "comparison_all_turbulence_models_newMesh", "3.5deg")
cases = ["URANS_SA_alpha3.5deg_blockMesh_newMesh", "URANS_kOmegaSST_alpha3.5deg_blockMesh_newMesh",
         "URANS_SALSA_alpha3.5deg_blockMesh_useRmod_useSmod_newMesh", "URANS_SA_rc_alpha3.5deg_blockMesh_newMesh",
         "URANS_RSM_SSG_alpha3.5deg_blockMesh_newMesh_couplingIter_0", "URANS_RSM_LRR_alpha3.5deg_blockMesh_newMesh_couplingIter_0"]

legend = [r"$\mathrm{SA}$", r"$\mathrm{k-}\omega \mathrm{~SST}$", r"$\mathrm{SALSA}$", r"$\mathrm{SA-rc}$",
          r"$\mathrm{SSG}~\mathrm{(RSM)}$", r"$\mathrm{LRR}~\mathrm{(RSM)}$"]

In [ ]:
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation", "URANS_2D_Ma0.73_Re3e6")
save_dir = join("..", "run", "plots", "URANS_validation", "URANS_blockMesh", "comparison_RSM_newMesh", "3.5deg")
cases = ["URANS_RSM_SSG_alpha3.5deg_blockMesh_newMesh_couplingIter_0", "URANS_RSM_SSG_alpha3.5deg_blockMesh_newMesh_couplingIter_1",
         "URANS_RSM_LRR_alpha3.5deg_blockMesh_newMesh_couplingIter_0", "URANS_RSM_LRR_alpha3.5deg_blockMesh_newMesh_couplingIter_1"]

legend = [r"$\mathrm{SSG}~\mathrm{(couplingIter~0)}$", r"$\mathrm{SSG}~\mathrm{(couplingIter~1)}$",
          r"$\mathrm{LRR}~\mathrm{(couplingIter~0)}$", r"$\mathrm{LRR}~\mathrm{(couplingIter~1)}$"]

In [ ]:
field = "total(p)_coeff"
save_name = "cp"
xy = [False, False, False, False, False, False]          # orientation (plane) of the airfoil

# path to the experimental data from Jacquin et al (MA = 0.73, alpha = 2.5 deg & 3.5 deg)
exp_path = [join("..", "validation_exp_data", f"Jacquin_cp_Ma0.73_alpha{alpha}deg.csv") for alpha in aoa]
use_exp_data = True

In [ ]:
# create plot directory
if not exists(save_dir):
    makedirs(save_dir)

In [ ]:
# load the experimental data if specified
if use_exp_data:
    exp_data = [read_csv(p, sep=",", comment="#", header=None, usecols=[0, 1], names=["x", "cp"]) for p in exp_path]

In [ ]:
x, z, cp, camber, x_camber = [], [], [], [], []
counter = 0
# TODO: put in separate function -> we also have to avg. over spanwise direction (e.g. for DDES)
for i, c in enumerate(cases):
        loader = CSVDataloader.from_foam_surface(join(load_dir, c, "postProcessing", "surface"), f"{field}_airfoil.raw")
        x_temp = loader.vertices[:, 0]

        # new mesh is oriented in the x-y-plane or in the x-y-plane depending on the setup
        if xy[i]:
            z_temp = loader.vertices[:, 1]
        else:
            z_temp = loader.vertices[:, 2]
        x_camber_temp, camber_line = compute_camber_line(x_temp, c=1, xf_max=0.5, f_max=0.05)

        # get all coordinates for suction and pressure side
        is_suction = z_temp > camber_line
        is_pressure = ~is_suction

        # take all times starting at t = XX to compute the mean cp if we have URANS. If the last time < t_start just use the last few
        write_times = [t for t in loader.write_times if float(t) >= t_start]
        if not write_times:
            write_times = loader.write_times[-25:]

        # for URANS we want to avg. therefore we have to adjust write_times
        cp_temp = loader.load_snapshot(field, write_times).unsqueeze(-1)
        cp_suction = cp_temp[is_suction]
        cp_pressure = cp_temp[is_pressure]

        x_suction = x_temp[is_suction]
        z_suction = z_temp[is_suction]
        x_pressure = x_temp[is_pressure]
        z_pressure = z_temp[is_pressure]

        idx_suction = pt.argsort(x_suction, descending=True)
        x_suction = x_suction[idx_suction]
        z_suction = z_suction[idx_suction]
        cp_suction = cp_suction[idx_suction]

        idx_pressure = pt.argsort(x_pressure)
        x_pressure = x_pressure[idx_pressure]
        z_pressure = z_pressure[idx_pressure]
        cp_pressure = cp_pressure[idx_pressure]

        x_sorted = pt.cat([x_suction, x_pressure])
        z_sorted = pt.cat([z_suction, z_pressure])
        cp_sorted = pt.cat([cp_suction, cp_pressure])

        x.append(x_sorted)
        z.append(z_sorted)
        cp.append(cp_sorted)
        camber.append(camber_line)
        x_camber.append(x_camber_temp)
        counter += 1

In [ ]:
# plot mean cp-distribution
fig, ax = plt.subplots(figsize=(6, 4))
for j in range(len(cases)):
    ax.plot(x[j]/chord, cp[j].mean(dim=1), zorder=10, color=color[j], label=f"{legend[j]}")

# add experimental data if specified
if use_exp_data:
    for a in range(len(aoa)):
        ax.scatter(exp_data[a]["x"][::10], exp_data[a]["cp"][::10], zorder=10, color="black", marker="o" if aoa[a] == 2.5 else "x", facecolor="none" if aoa[a] == 2.5 else "black",
                       label=r"$\mathrm{Jacquin}~\mathrm{et}~\mathrm{al.}~(\alpha = " + str(aoa[a]) + r"^\circ)$")
                       # label=r"$\mathrm{Jacquin}~\mathrm{et}~\mathrm{al.}$")

ax.set_xlabel(r"$x/c$")
ax.set_ylabel("$c_p$")
ax.grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
plt.gca().invert_yaxis()
fig.tight_layout()
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.8)
plt.savefig(join(save_dir, f"comparison_mean_{save_name}_distribution.png"))

In [ ]:
# plot RSM(p) and compare to experimental data
pRSM_exp = read_csv(join("..", "validation_exp_data", f"Jacquin_pRMS_Ma0.73_alpha3.5deg.csv"), sep=",", comment="#", header=None, usecols=[0, 1], names=["x", "pRMS"])

fig, ax = plt.subplots(figsize=(6, 3))
for j in range(len(cases)):
    # compute RMS
    pRMS = (cp[j] - cp[j].mean(dim=1).unsqueeze(-1)).pow(2).mean(1).sqrt()

    # plot RMS of SS only
    ax.plot(x[j][:x[j].size(0)//2]/chord, pRMS[:x[j].size(0)//2], zorder=10, color=color[j], label=f"{legend[j]}")

# add plot for experimental data
plt.plot(pRSM_exp["x"], pRSM_exp["pRMS"], zorder=10, color="black", marker="s", fillstyle="none", label=r"$\mathrm{Jacquin}~\mathrm{et}~\mathrm{al.}~(\alpha = 3.5^\circ)$")
# plt.plot(pRSM_exp["x"], pRSM_exp["pRMS"], zorder=10, color="black", marker="s", fillstyle="none", label=r"$\mathrm{Jacquin}~\mathrm{et}~\mathrm{al.}$")

ax.set_xlim(0.2, 1)
ax.set_xlabel(r"$x/c$")
ax.set_ylabel(r"$p^\prime_\mathrm{RMS} / q_\infty$")
ax.grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
fig.tight_layout()
fig.legend(ncol=3, loc="upper center")
fig.subplots_adjust(top=0.78)
plt.savefig(join(save_dir, f"comparison_pRMS.png"))


In [ ]:
# plot cp-distribution at different points in time TODO: refactor etc.
fig, ax = plt.subplots(figsize=(6, 4))
for j in range(len(cases)):
    for i in range(cp[j].size(-1)):
        ax.plot(x[j]/chord, cp[j][:, i], ls=ls[i], zorder=10, color=color[j],
                label=rf"{legend[j]} $(t^* = " + "{:.0f}".format(float(write_times[i]) * u_inf / chord) + ")$")
ax.set_xlabel(r"$x/c$")
ax.set_ylabel("$c_p$")
ax.grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
ax.minorticks_on()
ax.tick_params(axis="x", which="minor", bottom=False)
ax.grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
plt.gca().invert_yaxis()
fig.tight_layout()
fig.legend(ncol=2, loc="upper center")
fig.subplots_adjust(top=0.8)
# plt.savefig(join(save_dir, f"comparison_{save_name}_distribution.png"))

In [ ]:
# test plot airfoil with computed camber line
fig, ax = plt.subplots(figsize=(6, 2))

# AF from dat file
ax.plot(x[0], z[0], color=color[0], label="OAT15A", ls="--")
ax.scatter(x_camber[0], camber[0], color=color[1], marker=".", s=2)
ax.set_aspect("equal")
ax.set_xlabel("$x / c$")
ax.set_ylabel("$z / c$")
fig.tight_layout()
fig.legend(ncols=2, loc="upper center")
plt.savefig(join(save_dir, "airfoil.png"))
plt.show()